In [46]:
import pandas as pd
import numpy as np

from keras.models import load_model
import joblib
import time

In [47]:
# Load scaler dan model
scaler = joblib.load("scaler.pkl")
model = load_model("best_model.keras")

# Urutan fitur HARUS sama dengan saat training
features = ['moisture_before', 'moisture_after', 'duration', 'moisture_gain', 'moisture_rate']
mse_threshold = 0.6523

In [48]:
def irrigationDetection(data: dict):
    time_start = int(time.time() * 1000)

    df = pd.DataFrame([data], columns=features)

    scaler_df = scaler.transform(df)

    recon_data = model.predict(scaler_df, verbose=0)

    error_data = np.abs(scaler_df - recon_data)

    mean_error = np.mean(error_data)
    fault_ratio = mean_error / mse_threshold

    dominant_index = np.argmax(error_data)
    dominant_feature = features[dominant_index]

    max_error = np.max(error_data)

    prediction_time = int(time.time() * 1000) - time_start

    print('Prediction Dilakukan')

    return {
        "avg_mse": round(mean_error, 2),
        "fault_ratio": round(fault_ratio, 2),
        "flag": fault_ratio >= 1,
        "severity": getSeverity(fault_ratio),

        "dominant_feature": dominant_feature,
        "dominant_ratio": round(max_error / np.sum(error_data), 3),
        "dominant_error": round(max_error, 2),

        "mse_moisture_before": round(error_data[0][0], 2),
        "mse_moisture_after": round(error_data[0][1], 2),
        "mse_duration": round(error_data[0][2], 2),
        "mse_moisture_gain": round(error_data[0][3], 2),
        "mse_moisture_rate": round(error_data[0][4], 2),

        "prediction_time": prediction_time,
        "time": int(time.time() * 1000),
    }


def getSeverity(fault_ratio: float):
    if fault_ratio > 2.5:
        return "high"
    elif fault_ratio > 1.5:
        return "medium"
    else:
        return "low"

In [49]:
# Data Normal
moisture_before = 36
moisture_after = 79.6
moisture_gain = moisture_after - moisture_before
duration = 6.61

dict_event = {
        "moisture_before": moisture_before,
        "moisture_after": moisture_after,
        "duration": duration,
        "moisture_gain": moisture_gain,
        "moisture_rate": (moisture_gain) / duration,
}

In [50]:
result = irrigationDetection(dict_event)
result

Prediction Dilakukan


{'avg_mse': np.float64(0.14),
 'fault_ratio': np.float64(0.22),
 'flag': np.False_,
 'severity': 'low',
 'dominant_feature': 'moisture_after',
 'dominant_ratio': np.float64(0.326),
 'dominant_error': np.float64(0.23),
 'mse_moisture_before': np.float64(0.22),
 'mse_moisture_after': np.float64(0.23),
 'mse_duration': np.float64(0.13),
 'mse_moisture_gain': np.float64(0.05),
 'mse_moisture_rate': np.float64(0.08),
 'prediction_time': 205,
 'time': 1783658104761}